In [1]:
import osmnx as ox

import pandas as pd
import numpy as np

from pathlib import Path

import requests
import json

from rapidfuzz import fuzz
from unidecode import unidecode

# Load các street type

In [2]:
streets_df = pd.read_csv("../data/raw/streets.csv")
streets_types = streets_df["type"].unique().tolist()
streets_types.remove("unclassified")
print(streets_types)

['trunk', 'tertiary', 'secondary', 'primary_link', 'primary', 'trunk_link', 'tertiary_link', 'secondary_link', 'motorway_link', 'motorway']


# Load graph

In [3]:
with open("../data/raw/osm_train_2019_01_03.json", "r", encoding="utf-8") as f:
    osm_data = json.load(f)

In [4]:
osm_data.keys()

dict_keys(['version', 'generator', 'osm3s', 'elements'])

In [5]:
osm_data["version"]

0.6

In [6]:
osm_data["osm3s"]

{'timestamp_osm_base': '2026-05-21T03:31:08Z',
 'copyright': 'The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.'}

In [7]:
osm_elements = osm_data["elements"]

In [8]:
osm_elements_df = pd.json_normalize(osm_elements)
osm_elements_df.head()

,type,id,lat,lon,tags.highway,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,tags.name,...,tags.name:ja,tags.name:th,tags.fixme,tags.lay,tags.addr:place,tags.information,tags.ferry,tags.covered,tags.motorcar:forward,tags.footway
0,node,366367322,10.799155,106.657136,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,366367392,10.775732,106.614032,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,366367450,10.753841,106.645673,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,366367451,10.793792,106.695366,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,366367839,10.807689,106.664528,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
osm_elements_df["type"].unique()

array(['node', 'way', 'relation'], dtype=object)

In [10]:
osm_elements_df.columns

Index(['type', 'id', 'lat', 'lon', 'tags.highway', 'tags.railway',
       'tags.traffic_signals', 'tags.barrier', 'tags.junction', 'tags.name',
       ...
       'tags.name:ja', 'tags.name:th', 'tags.fixme', 'tags.lay',
       'tags.addr:place', 'tags.information', 'tags.ferry', 'tags.covered',
       'tags.motorcar:forward', 'tags.footway'],
      dtype='object', length=130)

In [11]:
non_tags_attrs = [x for x in osm_elements_df.columns if not x.startswith("tags")]
non_tags_attrs

['type', 'id', 'lat', 'lon', 'nodes', 'members']

# OSM Node

In [12]:
osm_nodes_full_df = osm_elements_df[
    osm_elements_df["type"] == "node"
].dropna(axis=1, how="all")

print(osm_nodes_full_df.shape)
osm_nodes_full_df.head()

(53115, 59)


,type,id,lat,lon,tags.highway,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,tags.name,...,tags.name:pt,tags.fee,tags.parking,tags.button_operated,tags.traffic_signals:sound,tags.phone,tags.website,tags.addr:place,tags.information,tags.ferry
0,node,366367322,10.799155,106.657136,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,366367392,10.775732,106.614032,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,366367450,10.753841,106.645673,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,366367451,10.793792,106.695366,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,366367839,10.807689,106.664528,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
(osm_nodes_full_df.isnull().sum() * 100 / len(osm_nodes_full_df)).sort_values(ascending=True)

type                           0.000000
id                             0.000000
lat                            0.000000
lon                            0.000000
tags.highway                  97.686153
tags.name                     98.456180
tags.note                     99.638520
tags.operator                 99.873859
tags.traffic_signals          99.885155
tags.addr:housenumber         99.887038
tags.addr:street              99.888920
tags.barrier                  99.919044
tags.railway                  99.945401
tags.name:en                  99.947284
tags.crossing                 99.949167
tags.crossing_ref             99.964229
tags.amenity                  99.977408
tags.shelter                  99.983056
tags.name:ru                  99.986821
tags.name:vi                  99.986821
tags.public_transport         99.988704
tags.addr:city                99.992469
tags.addr:district            99.992469
tags.access                   99.992469
tags.entrance                 99.992469


In [15]:
osm_nodes_df = osm_elements_df[osm_elements_df["type"] == "node"][non_tags_attrs]
print(osm_nodes_df.shape)
osm_nodes_df.head()

(53115, 6)


,type,id,lat,lon,nodes,members
0,node,366367322,10.799155,106.657136,NaN,NaN
1,node,366367392,10.775732,106.614032,NaN,NaN
2,node,366367450,10.753841,106.645673,NaN,NaN
3,node,366367451,10.793792,106.695366,NaN,NaN
4,node,366367839,10.807689,106.664528,NaN,NaN


In [16]:
osm_nodes_df.isnull().sum() * 100 / len(osm_nodes_df)

type         0.0
id           0.0
lat          0.0
lon          0.0
nodes      100.0
members    100.0
dtype: float64

In [17]:
osm_nodes_df = osm_nodes_df.drop(columns=["nodes", "members"])
osm_nodes_df.head()

,type,id,lat,lon
0,node,366367322,10.799155,106.657136
1,node,366367392,10.775732,106.614032
2,node,366367450,10.753841,106.645673
3,node,366367451,10.793792,106.695366
4,node,366367839,10.807689,106.664528


In [18]:
node_tags = sorted([x for x in osm_nodes_full_df.columns if x.startswith("tags.")])
left_tags = node_tags.copy()
print(len(node_tags))
print(node_tags)

55
['tags.access', 'tags.addr:city', 'tags.addr:district', 'tags.addr:housename', 'tags.addr:housenumber', 'tags.addr:place', 'tags.addr:postcode', 'tags.addr:street', 'tags.addr:subdistrict', 'tags.amenity', 'tags.barrier', 'tags.bench', 'tags.building', 'tags.bus', 'tags.button_operated', 'tags.crossing', 'tags.crossing_ref', 'tags.cuisine', 'tags.diet:vegan', 'tags.diet:vegetarian', 'tags.entrance', 'tags.fee', 'tags.ferry', 'tags.highway', 'tags.highway_1', 'tags.information', 'tags.int_name', 'tags.junction', 'tags.layer', 'tags.leisure', 'tags.name', 'tags.name:de', 'tags.name:en', 'tags.name:es', 'tags.name:pt', 'tags.name:ru', 'tags.name:vi', 'tags.network', 'tags.noexit', 'tags.note', 'tags.operator', 'tags.parking', 'tags.phone', 'tags.public_transport', 'tags.railway', 'tags.ref', 'tags.shelter', 'tags.shop', 'tags.supervised', 'tags.tactile_paving', 'tags.tourism', 'tags.traffic_signals', 'tags.traffic_signals:sound', 'tags.website', 'tags.wheelchair']


In [19]:
node_addr_tags = [x for x in node_tags if x.startswith("tags.addr")]
left_tags = [x for x in left_tags if x not in node_addr_tags]
print(len(node_addr_tags))
print(len(left_tags))
print(left_tags)

8
47
['tags.access', 'tags.amenity', 'tags.barrier', 'tags.bench', 'tags.building', 'tags.bus', 'tags.button_operated', 'tags.crossing', 'tags.crossing_ref', 'tags.cuisine', 'tags.diet:vegan', 'tags.diet:vegetarian', 'tags.entrance', 'tags.fee', 'tags.ferry', 'tags.highway', 'tags.highway_1', 'tags.information', 'tags.int_name', 'tags.junction', 'tags.layer', 'tags.leisure', 'tags.name', 'tags.name:de', 'tags.name:en', 'tags.name:es', 'tags.name:pt', 'tags.name:ru', 'tags.name:vi', 'tags.network', 'tags.noexit', 'tags.note', 'tags.operator', 'tags.parking', 'tags.phone', 'tags.public_transport', 'tags.railway', 'tags.ref', 'tags.shelter', 'tags.shop', 'tags.supervised', 'tags.tactile_paving', 'tags.tourism', 'tags.traffic_signals', 'tags.traffic_signals:sound', 'tags.website', 'tags.wheelchair']


In [20]:
node_name_tags = [x for x in node_tags if x.startswith("tags.name")]
left_tags = [x for x in left_tags if x not in node_name_tags]
print(len(node_name_tags))
print(len(left_tags))
print(left_tags)

7
40
['tags.access', 'tags.amenity', 'tags.barrier', 'tags.bench', 'tags.building', 'tags.bus', 'tags.button_operated', 'tags.crossing', 'tags.crossing_ref', 'tags.cuisine', 'tags.diet:vegan', 'tags.diet:vegetarian', 'tags.entrance', 'tags.fee', 'tags.ferry', 'tags.highway', 'tags.highway_1', 'tags.information', 'tags.int_name', 'tags.junction', 'tags.layer', 'tags.leisure', 'tags.network', 'tags.noexit', 'tags.note', 'tags.operator', 'tags.parking', 'tags.phone', 'tags.public_transport', 'tags.railway', 'tags.ref', 'tags.shelter', 'tags.shop', 'tags.supervised', 'tags.tactile_paving', 'tags.tourism', 'tags.traffic_signals', 'tags.traffic_signals:sound', 'tags.website', 'tags.wheelchair']


In [21]:
left_tags

['tags.access',
 'tags.amenity',
 'tags.barrier',
 'tags.bench',
 'tags.building',
 'tags.bus',
 'tags.button_operated',
 'tags.crossing',
 'tags.crossing_ref',
 'tags.cuisine',
 'tags.diet:vegan',
 'tags.diet:vegetarian',
 'tags.entrance',
 'tags.fee',
 'tags.ferry',
 'tags.highway',
 'tags.highway_1',
 'tags.information',
 'tags.int_name',
 'tags.junction',
 'tags.layer',
 'tags.leisure',
 'tags.network',
 'tags.noexit',
 'tags.note',
 'tags.operator',
 'tags.parking',
 'tags.phone',
 'tags.public_transport',
 'tags.railway',
 'tags.ref',
 'tags.shelter',
 'tags.shop',
 'tags.supervised',
 'tags.tactile_paving',
 'tags.tourism',
 'tags.traffic_signals',
 'tags.traffic_signals:sound',
 'tags.website',
 'tags.wheelchair']

In [22]:
node_info_tags = node_name_tags + node_addr_tags + [
    "tags.website",
    "tags.ref",
    "tags.note",
    "tags.email",
    "tags.description",
    "tags.phone",
    "tags.operator",
    "tags.information",
]

In [23]:
left_tags = [x for x in left_tags if x not in node_info_tags]
print(len(left_tags))
left_tags

34


['tags.access',
 'tags.amenity',
 'tags.barrier',
 'tags.bench',
 'tags.building',
 'tags.bus',
 'tags.button_operated',
 'tags.crossing',
 'tags.crossing_ref',
 'tags.cuisine',
 'tags.diet:vegan',
 'tags.diet:vegetarian',
 'tags.entrance',
 'tags.fee',
 'tags.ferry',
 'tags.highway',
 'tags.highway_1',
 'tags.int_name',
 'tags.junction',
 'tags.layer',
 'tags.leisure',
 'tags.network',
 'tags.noexit',
 'tags.parking',
 'tags.public_transport',
 'tags.railway',
 'tags.shelter',
 'tags.shop',
 'tags.supervised',
 'tags.tactile_paving',
 'tags.tourism',
 'tags.traffic_signals',
 'tags.traffic_signals:sound',
 'tags.wheelchair']

## Primary Features

In [24]:
print(osm_nodes_full_df["tags.amenity"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.amenity"].notna()].shape)

[nan 'pharmacy' 'shelter' 'fuel' 'marketplace' 'restaurant' 'bank'
 'parking_entrance' 'Dai hoc Bach khoa TPHCM' 'ferry_terminal']
(12, 59)


In [26]:
print(osm_nodes_full_df["tags.building"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.building"].notna()].shape)

[nan 'church' 'office']
(2, 59)


In [27]:
print(osm_nodes_full_df["tags.shop"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.shop"].notna()].shape)

[nan 'clothes' 'department_store']
(2, 59)


In [29]:
print(osm_nodes_full_df["tags.entrance"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.entrance"].notna()].shape)

[nan 'yes' 'main']
(4, 59)


## Connectivity

In [86]:
node_connectivity_tags = [
    "tags.railway",
    "tags.junction",
    "tags.crossing",
    "tags.highway"
]

In [87]:
print(osm_nodes_full_df["tags.junction"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.junction"].notna()].shape)

[nan 'yes']
(3, 59)


In [88]:
print(osm_nodes_full_df["tags.crossing"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.crossing"].notna()].shape)

[nan 'zebra' 'traffic_signals' 'uncontrolled']
(27, 59)


In [89]:
print(osm_nodes_full_df["tags.highway"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.highway"].notna()].shape)

[nan 'traffic_signals' 'bus_stop' 'crossing' 'stop' 'turning_circle']
(1229, 59)


In [90]:
print(osm_nodes_full_df["tags.railway"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.railway"].notna()].shape)

[nan 'level_crossing' 'station' 'crossing']
(29, 59)


## Transportation

In [30]:
print(osm_nodes_full_df["tags.highway"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.highway"].notna()].shape)

[nan 'traffic_signals' 'bus_stop' 'crossing' 'stop' 'turning_circle']
(1229, 59)


In [55]:
print(osm_nodes_full_df["tags.highway_1"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.highway_1"].notna()].shape)

[nan 'traffic_signals']
(1, 59)


In [31]:
print(osm_nodes_full_df["tags.traffic_signals"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.traffic_signals"].notna()].shape)

[nan 'signal']
(61, 59)


In [32]:
print(osm_nodes_full_df["tags.crossing"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.crossing"].notna()].shape)

[nan 'zebra' 'traffic_signals' 'uncontrolled']
(27, 59)


In [33]:
print(osm_nodes_full_df["tags.barrier"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.barrier"].notna()].shape)

[nan 'toll_booth' 'gate' 'lift_gate' 'yes']
(43, 59)


In [35]:
print(osm_nodes_full_df["tags.bus"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.bus"].notna()].shape)

[nan 'yes']
(3, 59)


In [36]:
osm_nodes_full_df["tags.ferry"].unique()
print(osm_nodes_full_df[osm_nodes_full_df["tags.ferry"].notna()].shape)

(2, 59)


In [45]:
print(osm_nodes_full_df[osm_nodes_full_df["tags.highway"]=="bus_stop"].shape)

(833, 59)


In [47]:
osm_nodes_full_df["tags.highway"].value_counts()

tags.highway
bus_stop           833
traffic_signals    346
crossing            47
stop                 2
turning_circle       1
Name: count, dtype: int64

In [54]:
osm_nodes_full_df[
    (osm_nodes_full_df["tags.highway"] == "crossing")  &
    (osm_nodes_full_df["tags.crossing"] == "yes")
].shape

(0, 59)

In [57]:
print(osm_nodes_full_df["tags.railway"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.railway"].notna()].shape)

[nan 'level_crossing' 'station' 'crossing']
(29, 59)


In [60]:
print(osm_nodes_full_df["tags.public_transport"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.public_transport"].notna()].shape)

[nan 'station' 'stop_position' 'platform']
(6, 59)


## Public transportation

In [61]:
node_public_tags = [
    "tags.bus",
    "tags.ferry",
]

In [62]:
print(osm_nodes_full_df["tags.bus"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.bus"].notna()].shape)

[nan 'yes']
(3, 59)


In [63]:
osm_nodes_full_df["tags.ferry"].unique()
print(osm_nodes_full_df[osm_nodes_full_df["tags.ferry"].notna()].shape)

(2, 59)


# OSM Way

In [64]:
osm_ways_full_df = osm_elements_df[
    osm_elements_df["type"] == "way"
].dropna(axis=1, how="all")
osm_ways_full_df.to_csv("../data/preprocess/osm_ways_full.csv", index=False)
print(osm_ways_full_df.shape)
osm_ways_full_df.head()

(7606, 90)


,type,id,tags.highway,tags.traffic_signals,tags.junction,tags.name,tags.note,tags.addr:street,tags.addr:housenumber,tags.name:vi,...,tags.minspeed,tags.maxweight,tags.name:ja,tags.name:th,tags.fixme,tags.lay,tags.information,tags.covered,tags.motorcar:forward,tags.footway
3061,way,32575768,residential,NaN,NaN,Đường số 27,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3062,way,32576350,residential,NaN,NaN,Đường số 18,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3063,way,32576691,secondary,NaN,NaN,Phan Văn Hớn,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3064,way,32576911,residential,NaN,NaN,Tân Thành,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3065,way,32577060,residential,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
(osm_ways_full_df.isnull().sum() * 100 / len(osm_ways_full_df)).sort_values(ascending=True)

type                      0.000000
id                        0.000000
tags.highway              0.000000
nodes                     0.000000
tags.name                38.246121
                           ...    
tags.fixme               99.986852
tags.lay                 99.986852
tags.covered             99.986852
tags.motorcar:forward    99.986852
tags.footway             99.986852
Length: 90, dtype: float64

In [66]:
osm_ways_df = osm_elements_df[osm_elements_df["type"] == "way"][non_tags_attrs]
print(osm_ways_df.shape)
osm_ways_df.head()

(7606, 6)


,type,id,lat,lon,nodes,members
3061,way,32575768,NaN,NaN,"[366369613, 5795144851, 366418963, 366373068, ...",NaN
3062,way,32576350,NaN,NaN,"[366372346, 5755079612, 3040292134, 5755079614...",NaN
3063,way,32576691,NaN,NaN,"[366452320, 3351962143, 3351962141, 366375776,...",NaN
3064,way,32576911,NaN,NaN,"[5778381635, 5552002921, 4878713065, 580288880...",NaN
3065,way,32577060,NaN,NaN,"[366371080, 366389438, 5735589498, 5735589492,...",NaN


In [67]:
osm_ways_df.isnull().sum() * 100 / len(osm_ways_df)

type         0.0
id           0.0
lat        100.0
lon        100.0
nodes        0.0
members    100.0
dtype: float64

In [68]:
osm_ways_df = osm_ways_df.drop(columns=["lat", "lon", "members"])
osm_ways_df.head()

,type,id,nodes
3061,way,32575768,"[366369613, 5795144851, 366418963, 366373068, ..."
3062,way,32576350,"[366372346, 5755079612, 3040292134, 5755079614..."
3063,way,32576691,"[366452320, 3351962143, 3351962141, 366375776,..."
3064,way,32576911,"[5778381635, 5552002921, 4878713065, 580288880..."
3065,way,32577060,"[366371080, 366389438, 5735589498, 5735589492,..."


In [69]:
osm_way_tags = sorted([x for x in osm_ways_full_df.columns if x.startswith("tags.")])
left_tags = osm_way_tags.copy()
len(osm_way_tags)

87

In [70]:
left_tags

['tags.access',
 'tags.addr:city',
 'tags.addr:district',
 'tags.addr:housenumber',
 'tags.addr:postcode',
 'tags.addr:province',
 'tags.addr:street',
 'tags.addr:street:name',
 'tags.addr:subdistrict',
 'tags.alt_name',
 'tags.amenity',
 'tags.area',
 'tags.bicycle',
 'tags.bicycle:forward',
 'tags.bicycle:oneway',
 'tags.bridge',
 'tags.bridge:name',
 'tags.bridge:name:en',
 'tags.bridge_name',
 'tags.bridge_name:en',
 'tags.bus',
 'tags.car',
 'tags.construction',
 'tags.covered',
 'tags.crossing',
 'tags.cycleway',
 'tags.description',
 'tags.ele',
 'tags.fixme',
 'tags.foot',
 'tags.footway',
 'tags.hgv',
 'tags.highway',
 'tags.horse',
 'tags.information',
 'tags.int_name',
 'tags.int_ref',
 'tags.junction',
 'tags.lanes',
 'tags.lay',
 'tags.layer',
 'tags.level',
 'tags.lit',
 'tags.maxheight',
 'tags.maxspeed',
 'tags.maxweight',
 'tags.minspeed',
 'tags.motor_vehicle',
 'tags.motorcar',
 'tags.motorcar:backward',
 'tags.motorcar:forward',
 'tags.motorcycle',
 'tags.motorcycle

In [100]:
way_name_tags = [
    x for x in osm_way_tags if 
        x.startswith("tags.name") or 
        x.startswith("tags.old_name") or
        x.startswith("tags.alt_name")
]
left_tags = [x for x in left_tags if x not in way_name_tags]
print(len(way_name_tags))
print(way_name_tags)

14
['tags.alt_name', 'tags.name', 'tags.name:de', 'tags.name:en', 'tags.name:ja', 'tags.name:th', 'tags.name:vi', 'tags.name:vi-hani', 'tags.name:zh', 'tags.old_name', 'tags.old_name:en', 'tags.old_name:fr', 'tags.old_name:vi', 'tags.old_name:zh']


In [101]:
way_addr_tags = [x for x in osm_way_tags if x.startswith("tags.addr")]
left_tags = [x for x in left_tags if x not in way_addr_tags]
print(len(way_addr_tags))
print(way_addr_tags)

8
['tags.addr:city', 'tags.addr:district', 'tags.addr:housenumber', 'tags.addr:postcode', 'tags.addr:province', 'tags.addr:street', 'tags.addr:street:name', 'tags.addr:subdistrict']


In [102]:
way_lane_tags = [x for x in left_tags if x.startswith("tags.lane")]
left_tags = [x for x in left_tags if x not in way_lane_tags]
print(len(way_lane_tags))
print(way_lane_tags)

1
['tags.lanes']


In [103]:
way_bicycle_tags = [x for x in left_tags if x.startswith("tags.bicycle")]
left_tags = [x for x in left_tags if x not in way_bicycle_tags]
print(len(way_bicycle_tags))
print(way_bicycle_tags)

3
['tags.bicycle', 'tags.bicycle:forward', 'tags.bicycle:oneway']


In [104]:
way_maxspeed_tags = [x for x in left_tags if x.startswith("tags.maxspeed")]
left_tags = [x for x in left_tags if x not in way_maxspeed_tags]
print(len(way_maxspeed_tags))
print(way_maxspeed_tags)

1
['tags.maxspeed']


In [105]:
way_oneway_tags = [x for x in left_tags if x.startswith("tags.oneway")]
left_tags = [x for x in left_tags if x not in way_oneway_tags]
print(len(way_oneway_tags))
print(way_oneway_tags)

4
['tags.oneway', 'tags.oneway:bicycle', 'tags.oneway:motorcar', 'tags.oneway:motorcycle']


In [106]:
way_maxweight_tags = [x for x in left_tags if x.startswith("tags.maxweight")]
left_tags = [x for x in left_tags if x not in way_maxweight_tags]
print(len(way_maxweight_tags))
print(way_maxweight_tags)

1
['tags.maxweight']


In [107]:
way_hgv_tags = [x for x in left_tags if x.startswith("tags.hgv")]
left_tags = [x for x in left_tags if x not in way_hgv_tags]
print(len(way_hgv_tags))
print(way_hgv_tags)

1
['tags.hgv']


In [108]:
way_bus_tags = [x for x in left_tags if x.startswith("tags.bus")]
left_tags = [x for x in left_tags if x not in way_bus_tags]
print(len(way_bus_tags))
print(way_bus_tags)

1
['tags.bus']


In [109]:
way_motorcycle_tags = [x for x in left_tags if x.startswith("tags.motorcycle")]
left_tags = [x for x in left_tags if x not in way_motorcycle_tags]
print(len(way_bus_tags))
print(way_motorcycle_tags)

1
['tags.motorcycle', 'tags.motorcycle:oneway']


In [110]:
way_motor_vehicle_tags = [x for x in left_tags if x.startswith("tags.motor_vehicle")]
left_tags = [x for x in left_tags if x not in way_motor_vehicle_tags]
print(len(way_motor_vehicle_tags))
print(way_motor_vehicle_tags)

1
['tags.motor_vehicle']


In [111]:
way_infos_tags = way_addr_tags + way_name_tags + [
    "tags.note",
    "tags.description",
    "tags.ref",
    "tags.source",
    "tags.source:maxspeed",
    "tags.int_ref",
    "tags.date",
    "tags.information"
]
left_tags = [x for x in left_tags if x not in way_infos_tags]

In [112]:
(osm_ways_full_df[left_tags].isnull().sum() * 100 / len(osm_ways_full_df)).sort_values(ascending=True)

tags.highway               0.000000
tags.service              71.995793
tags.foot                 94.977649
tags.layer                96.029450
tags.bridge               96.450171
tags.surface              97.304759
tags.motorroad            98.672101
tags.horse                98.908756
tags.access               99.290034
tags.junction             99.329477
tags.motorcar             99.342624
tags.construction         99.723902
tags.width                99.868525
tags.bridge_name          99.907967
tags.bridge_name:en       99.907967
tags.tunnel               99.921115
tags.lit                  99.934262
tags.cycleway             99.947410
tags.bridge:name          99.947410
tags.sidewalk             99.947410
tags.maxheight            99.947410
tags.area                 99.947410
tags.motorcar:backward    99.960557
tags.minspeed             99.960557
tags.ski                  99.960557
tags.stop                 99.960557
tags.snowmobile           99.960557
tags.wheelchair           99

In [113]:
way_topology_tags = [
    "tags.highway"
]

In [114]:
left_tags

['tags.access',
 'tags.amenity',
 'tags.area',
 'tags.bridge',
 'tags.bridge:name',
 'tags.bridge:name:en',
 'tags.bridge_name',
 'tags.bridge_name:en',
 'tags.car',
 'tags.construction',
 'tags.covered',
 'tags.crossing',
 'tags.cycleway',
 'tags.ele',
 'tags.fixme',
 'tags.foot',
 'tags.footway',
 'tags.highway',
 'tags.horse',
 'tags.int_name',
 'tags.junction',
 'tags.lay',
 'tags.layer',
 'tags.level',
 'tags.lit',
 'tags.maxheight',
 'tags.minspeed',
 'tags.motorcar',
 'tags.motorcar:backward',
 'tags.motorcar:forward',
 'tags.motorroad',
 'tags.proposed',
 'tags.religion',
 'tags.service',
 'tags.sidewalk',
 'tags.ski',
 'tags.snowmobile',
 'tags.stop',
 'tags.surface',
 'tags.traffic_signals',
 'tags.tunnel',
 'tags.wheelchair',
 'tags.width']

## Physical Quality

In [115]:
way_physical_tags = [
    #"tags.highway",
    "tags.surface"
]

In [116]:
print(osm_ways_full_df["tags.highway"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.highway"].notna()]))

['residential' 'secondary' 'tertiary' 'service' 'unclassified' 'primary'
 'trunk' 'secondary_link' 'primary_link' 'footway' 'trunk_link' 'cycleway'
 'tertiary_link' 'path' 'living_street' 'road' 'pedestrian' 'track'
 'proposed' 'steps' 'motorway_link' 'motorway' 'construction']
7606


In [117]:
print(osm_ways_full_df["tags.surface"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.surface"].notna()]))

[nan 'asphalt' 'paved' 'concrete' 'paving_stones' 'dirt' 'unpaved']
205


In [118]:
oh_physical_df = pd.get_dummies(
    osm_ways_full_df[["id"] + way_physical_tags], 
    columns=way_physical_tags,
)
oh_physical_df.head()

,id,tags.surface_asphalt,tags.surface_concrete,tags.surface_dirt,tags.surface_paved,tags.surface_paving_stones,tags.surface_unpaved
3061,32575768,False,False,False,False,False,False
3062,32576350,False,False,False,False,False,False
3063,32576691,False,False,False,False,False,False
3064,32576911,False,False,False,False,False,False
3065,32577060,False,False,False,False,False,False


## Structure

In [119]:
way_structure_tags = [
    "tags.layer",
    "tags.bridge",
    "tags.cutting",
    "tags.lanes",
    "tags.oneway"
]

In [120]:
print(osm_ways_full_df["tags.layer"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.layer"].notna()]))

[nan '1' '5' '3' '4' '2' '-1']
302


In [121]:
print(osm_ways_full_df["tags.bridge"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.bridge"].notna()]))

[nan 'yes']
270


In [122]:
print(osm_ways_full_df["tags.tunnel"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.tunnel"].notna()]))

[nan 'yes' 'building_passage']
6


In [123]:
print(osm_ways_full_df["tags.lanes"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.lanes"].notna()]))

[nan '2' '3' '5' '4' '1' '6' '8']
179


In [124]:
print(osm_ways_full_df["tags.oneway"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.oneway"].notna()]))

[nan 'yes' 'no' '-1' '-1;yes' 'no;yes']
2223


In [125]:
print(osm_ways_full_df["tags.oneway"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.oneway"] == "no;yes"]))

[nan 'yes' 'no' '-1' '-1;yes' 'no;yes']
3


## Control & Restriction

In [128]:
way_control_tags = [
    "tags.maxspeed",
    "tags.minspeed",
    "tags.maxweight",
    "tags.motorroad"
]

In [129]:
print(osm_ways_full_df["tags.maxspeed"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.maxspeed"].notna()]))

[nan '50' '80' '60' '40' '20' '10' '30' '70' '40,50,60' '45' '120' '100']
521


In [130]:
print(osm_ways_full_df["tags.minspeed"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.minspeed"].notna()]))

[nan '60']
3


In [131]:
print(osm_ways_full_df["tags.maxweight"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.maxweight"].notna()]))

[nan '13' '1.5']
2


In [132]:
print(osm_ways_full_df["tags.hgv"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.hgv"].notna()]))

[nan 'delivery' 'no']
4


In [133]:
print(osm_ways_full_df["tags.motorcar"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.motorcar"].notna()]))

[nan 'no' 'designated']
50


In [135]:
print(osm_ways_full_df["tags.stop"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.stop"].notna()]))

[nan 'All']
3


In [72]:
print(osm_ways_full_df["tags.access"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.access"].notna()]))

[nan 'no' 'yes' 'private' 'designated' 'permissive']
54


In [73]:
print(osm_ways_full_df["tags.motor_vehicle"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.motor_vehicle"].notna()]))

[nan 'yes' 'designated' 'no' 'private' 'permissive']
58


In [74]:
print(osm_ways_full_df["tags.motorcar"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.motorcar"].notna()]))

[nan 'no' 'designated']
50


In [75]:
print(osm_ways_full_df["tags.motorcycle"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.motorcycle"].notna()]))

[nan 'yes' 'no' 'designated']
326


In [76]:
print(osm_ways_full_df["tags.bus"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.bus"].notna()]))

[nan 'no']
1


In [77]:
print(osm_ways_full_df["tags.hgv"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.hgv"].notna()]))

[nan 'delivery' 'no']
4


In [78]:
print(osm_ways_full_df["tags.bicycle"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.bicycle"].notna()]))

[nan 'no' 'yes' 'private' 'permissive']
435


In [79]:
print(osm_ways_full_df["tags.foot"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.foot"].notna()]))

[nan 'no' 'yes' 'permissive']
382


In [80]:
print(osm_ways_full_df["tags.junction"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.junction"].notna()]))

[nan 'roundabout']
51


In [81]:
print(osm_ways_full_df["tags.area"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.area"].notna()]))

[nan 'no']
4


In [82]:
print(osm_ways_full_df["tags.stop"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.stop"].notna()]))

[nan 'All']
3


In [83]:
print(osm_ways_full_df["tags.car"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.car"].notna()]))

[nan 'destination']
1


# OSM Relation

In [138]:
osm_relation_df = osm_elements_df[osm_elements_df["type"] == "relation"][non_tags_attrs]
print(osm_relation_df.shape)
osm_relation_df.head()

(63, 6)


,type,id,lat,lon,nodes,members
5631,relation,2922023,NaN,NaN,NaN,"[{'type': 'node', 'ref': 366443898, 'role': 'v..."
5632,relation,3423384,NaN,NaN,NaN,"[{'type': 'node', 'ref': 2079964894, 'role': '..."
7974,relation,2907508,NaN,NaN,NaN,"[{'type': 'node', 'ref': 366476538, 'role': 'v..."
7975,relation,2914954,NaN,NaN,NaN,"[{'type': 'node', 'ref': 366462455, 'role': 'v..."
7976,relation,2970026,NaN,NaN,NaN,"[{'type': 'node', 'ref': 2079964991, 'role': '..."


In [139]:
osm_relation_df.isnull().sum() * 100 / len(osm_relation_df)

type         0.0
id           0.0
lat        100.0
lon        100.0
nodes      100.0
members      0.0
dtype: float64

In [140]:
osm_relation_df = osm_relation_df.drop(columns=["lat", "lon", "nodes"])
osm_relation_df.head()

,type,id,members
5631,relation,2922023,"[{'type': 'node', 'ref': 366443898, 'role': 'v..."
5632,relation,3423384,"[{'type': 'node', 'ref': 2079964894, 'role': '..."
7974,relation,2907508,"[{'type': 'node', 'ref': 366476538, 'role': 'v..."
7975,relation,2914954,"[{'type': 'node', 'ref': 366462455, 'role': 'v..."
7976,relation,2970026,"[{'type': 'node', 'ref': 2079964991, 'role': '..."


In [141]:
# Tách mỗi member thành một dòng
df_exploded = osm_relation_df.explode('members').reset_index(drop=True)

# Tách dict trong members thành nhiều cột
member_df = pd.json_normalize(df_exploded['members'])

# Kếtmember_dfhợp lại với thông tin relation
member_df = pd.concat([
    df_exploded[['type', 'id']].reset_index(drop=True), 
    member_df
], axis=1)

member_df.head()

,type,id,type,ref,role
0,relation,2922023,node,366443898,via
1,relation,2922023,way,220971019,from
2,relation,2922023,way,32580244,to
3,relation,3423384,node,2079964894,via
4,relation,3423384,way,255646465,from


In [142]:
member_group = member_df.groupby("id")

In [143]:
for count, (idx, group) in enumerate(member_group):
    if count == 9:
        break
    if len(group) > 2:
        print(group.to_string())
        print("_"* 50)

        type       id  type         ref  role
94  relation  2857515  node  1767203801   via
95  relation  2857515   way   165149818  from
96  relation  2857515   way   215167469    to
__________________________________________________
        type       id  type         ref  role
78  relation  2857516  node  1767203801   via
79  relation  2857516   way    32576371    to
80  relation  2857516   way   326848019  from
__________________________________________________
        type       id  type        ref  role
88  relation  2864377  node  735998170   via
89  relation  2864377   way   59339636  from
90  relation  2864377   way   59265376    to
__________________________________________________
       type       id  type        ref  role
6  relation  2907508  node  366476538   via
7  relation  2907508   way  219856946  from
8  relation  2907508   way  239030925    to
__________________________________________________
         type       id  type         ref  role
111  relation  2916615  n

In [144]:
member_df["role"].unique()

array(['via', 'from', 'to', ''], dtype=object)

In [145]:
osm_nodes_full_df[osm_nodes_full_df ["id"] == 2212447332]

,type,id,lat,lon,tags.highway,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,tags.name,...,tags.name:pt,tags.fee,tags.parking,tags.button_operated,tags.traffic_signals:sound,tags.phone,tags.website,tags.addr:place,tags.information,tags.ferry
16244,node,2212447332,10.705947,106.598051,traffic_signals,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [146]:
member_df.to_csv("../data/raw/osm_relation_2019_01_03", index=False)

In [148]:
osm_ways_df["tags.turn"]

KeyError: 'tags.turn'